# 07 - Merge de NASA POWER al dataset combinado (DETER + BDQueimadas + SISAM)
### Proyecto BIODIVERSITY-GUARD-Predict (1ACC0057 - Machine Learning)

## Que hace este notebook

Ultima fuente del proyecto. Se parte de los datasets ya validados del Notebook 5 (`dataset_alertas_final.csv`, `dataset_semanal_final.csv`, que ya tienen DETER + BDQueimadas + SISAM correctamente cruzados con la clave `(municipio_norm, uf)`) y se les agrega NASA POWER (temperatura, humedad relativa, viento).

| | Frente 2 (`dataset_alertas`) | Frente 1 (`dataset_semanal`) |
|---|---|---|
| Ventana de NASA POWER | 7 dias previos a la fecha exacta de cada alerta | Promedio de la semana calendario completa |

Igual que SISAM, NASA POWER es un producto de reanalisis/modelo con cobertura completa (425 municipios x 1096 dias, 0 nulos) - no hace falta construir una grilla de fechas, ya viene completo.

**IMPORTANTE:** NASA POWER se descargo por **centroide de municipio** (promedio de lat/lon de las alertas de DETER en ese municipio), no por nombre de lugar administrativo oficial. La clave de cruce sigue siendo `(municipio_norm, uf)`, igual que con las demas fuentes, porque el centroide ya se calculo agrupando por esa misma clave.

## Archivos de entrada requeridos
1. `dataset_alertas_final.csv` - salida del Notebook 5 (49,971 filas, con DETER+BDQueimadas+SISAM)
2. `dataset_semanal_final.csv` - salida del Notebook 5 (11,445 filas, con DETER+BDQueimadas+SISAM)
3. `nasa_power_limpio.csv` - salida del Notebook 6 (465,800 filas, a nivel de registro diario)

In [1]:
import pandas as pd
import numpy as np

## Paso 1: Cargar los 3 archivos

In [2]:
dataset_alertas = pd.read_csv("dataset_alertas_final.csv")
dataset_alertas["view_date"] = pd.to_datetime(dataset_alertas["view_date"])

dataset_semanal = pd.read_csv("dataset_semanal_final.csv")
dataset_semanal["semana"] = pd.to_datetime(dataset_semanal["semana"])

df_nasa = pd.read_csv("nasa_power_limpio.csv")
df_nasa["fecha"] = pd.to_datetime(df_nasa["fecha"])

assert "uf" in dataset_alertas.columns, "dataset_alertas no tiene columna 'uf' - usa la version corregida"
assert "uf" in dataset_semanal.columns, "dataset_semanal no tiene columna 'uf' - usa la version corregida"

print("dataset_alertas (antes de NASA POWER):", dataset_alertas.shape)
print("dataset_semanal (antes de NASA POWER):", dataset_semanal.shape)
print("NASA POWER (nivel registro diario):", df_nasa.shape)

dataset_alertas (antes de NASA POWER): (49971, 30)
dataset_semanal (antes de NASA POWER): (11445, 18)
NASA POWER (nivel registro diario): (465800, 6)


## Paso 2: Enriquecer `dataset_alertas` (Frente 2) con NASA POWER

### 2.1 Ventana movil de 7 dias hacia atras, por (municipio, estado)

In [4]:
cols_nasa = ["T2M", "RH2M", "WS2M"]

nasa_ordenado = df_nasa.sort_values(["municipio_norm", "uf", "fecha"]).set_index("fecha")

nasa_rolling = (
    nasa_ordenado.groupby(["municipio_norm", "uf"])[cols_nasa]
    .rolling("7D", min_periods=1).mean()
    .reset_index()
)

nasa_rolling = nasa_rolling.rename(columns={c: f"{c}_7d_previo" for c in cols_nasa})

print(nasa_rolling.shape)
nasa_rolling.head()

(465800, 6)


,municipio_norm,uf,fecha,T2M_7d_previo,RH2M_7d_previo,WS2M_7d_previo
0,ABAETETUBA,PA,2022-01-01,25.620000,92.180000,0.140
1,ABAETETUBA,PA,2022-01-02,25.710000,91.475000,0.125
2,ABAETETUBA,PA,2022-01-03,25.586667,92.136667,0.110
3,ABAETETUBA,PA,2022-01-04,25.552500,91.345000,0.125
4,ABAETETUBA,PA,2022-01-05,25.592000,91.226000,0.112


### 2.2 Unir a cada alerta individual (por fecha exacta + municipio + estado)

In [5]:
dataset_alertas = dataset_alertas.merge(
    nasa_rolling,
    left_on=["view_date", "municipio_norm", "uf"],
    right_on=["fecha", "municipio_norm", "uf"],
    how="left"
).drop(columns=["fecha"])

print("dataset_alertas (con NASA POWER):", dataset_alertas.shape)
assert dataset_alertas.shape[0] == 49971, f"ALERTA: cambio el numero de filas: {dataset_alertas.shape[0]}"

print("\nNivel_Riesgo_Amenaza se conserva intacto:")
print(dataset_alertas["Nivel_Riesgo_Amenaza"].value_counts())

cols_nasa_7d = [c for c in dataset_alertas.columns if c.endswith("_7d_previo") and c.split("_7d_previo")[0] in cols_nasa]
print("\nNulos en las nuevas columnas de NASA POWER:")
print(dataset_alertas[cols_nasa_7d].isnull().sum())

dataset_alertas (con NASA POWER): (49971, 33)

Nivel_Riesgo_Amenaza se conserva intacto:
Nivel_Riesgo_Amenaza
Moderado    32440
Alto         7743
Bajo         6483
Crítico      3305
Name: count, dtype: int64

Nulos en las nuevas columnas de NASA POWER:
T2M_7d_previo     0
RH2M_7d_previo    0
WS2M_7d_previo    0
dtype: int64


In [6]:
print("Muestra de 5 filas con las columnas nuevas de NASA POWER:")
print(dataset_alertas[["municipio_norm", "uf", "view_date", "Nivel_Riesgo_Amenaza"] + cols_nasa_7d].sample(5, random_state=42))

print("\nEstadisticas de las columnas nuevas:")
print(dataset_alertas[cols_nasa_7d].describe())

Muestra de 5 filas con las columnas nuevas de NASA POWER:
              municipio_norm  uf  view_date Nivel_Riesgo_Amenaza  \
41143     SAO FELIX DO XINGU  PA 2024-09-26             Moderado   
6887   PORTO ALEGRE DO NORTE  MT 2022-08-16             Moderado   
19599                VILHENA  RO 2023-08-25                 Bajo   
33393            PORTO VELHO  RO 2024-08-11             Moderado   
45194                MELGACO  PA 2024-10-14             Moderado   

       T2M_7d_previo  RH2M_7d_previo  WS2M_7d_previo  
41143      31.002857       49.387143        0.158571  
6887       30.004286       31.675714        0.038571  
19599      30.692857       44.398571        0.174286  
33393      30.110000       48.324286        0.180000  
45194      32.404286       52.315714        0.224286  

Estadisticas de las columnas nuevas:
       T2M_7d_previo  RH2M_7d_previo  WS2M_7d_previo
count   49971.000000    49971.000000    49971.000000
mean       28.780675       63.119894        0.183861
std   

In [12]:
dataset_alertas.head()

,id_alerta,gid,classname,view_date,publish_month,sensor,satellite,municipality,mun_geocod,uf,...,riesgo_fuego_promedio_7d_previo,pm10_reanalise_7d_previo,pm2_5_reanalise_7d_previo,o3_reanalise_7d_previo,no2_reanalise_7d_previo,co_reanalise_7d_previo,so2_reanalise_7d_previo,T2M_7d_previo,RH2M_7d_previo,WS2M_7d_previo
0,14825,21177_hist,DESMATAMENTO_CR,2022-01-01,2022-01-01,WFI,AMAZONIA-1,Manoel Urbano,1200344,AC,...,NaN,21.66,16.38,10.06,0.07,0.11,0.01,26.35,90.55,0.00
1,15607,94782_hist,DESMATAMENTO_CR,2022-01-01,2022-01-01,WFI,AMAZONIA-1,Sena Madureira,1200500,AC,...,NaN,22.32,17.07,11.40,0.07,0.11,0.02,26.75,87.22,0.02
2,10765,21159_hist,DESMATAMENTO_CR,2022-01-01,2022-01-01,WFI,CBERS-4A,Porto Velho,1100205,RO,...,NaN,18.13,14.24,15.99,0.18,0.12,0.15,25.70,95.07,0.07
3,41620,21150_hist,DESMATAMENTO_CR,2022-01-01,2022-01-01,WFI,CBERS-4A,Boca do Acre,1300706,AM,...,0.0,21.23,16.61,15.93,0.08,0.11,0.02,26.90,88.55,0.02
4,9017,21157_hist,DESMATAMENTO_CR,2022-01-01,2022-01-01,WFI,CBERS-4A,Porto Velho,1100205,RO,...,NaN,18.13,14.24,15.99,0.18,0.12,0.15,25.70,95.07,0.07


## Paso 3: Enriquecer `dataset_semanal` (Frente 1) con NASA POWER

In [7]:
df_nasa["semana"] = df_nasa["fecha"].dt.to_period("W").dt.start_time

nasa_semanal = (
    df_nasa
    .groupby(["semana", "municipio_norm", "uf"])[cols_nasa]
    .mean()
    .reset_index()
)

print(nasa_semanal.shape)
nasa_semanal.head()

(67150, 6)


,semana,municipio_norm,uf,T2M,RH2M,WS2M
0,2021-12-27,ABAETETUBA,PA,25.710,91.475,0.125
1,2021-12-27,ABEL FIGUEIREDO,PA,25.575,86.185,1.570
2,2021-12-27,ACAILANDIA,MA,24.835,89.785,0.525
3,2021-12-27,ACARA,PA,25.470,91.160,0.185
4,2021-12-27,ACRELANDIA,AC,26.470,89.220,0.000


In [8]:
dataset_semanal = dataset_semanal.merge(
    nasa_semanal,
    on=["semana", "municipio_norm", "uf"],
    how="left"
)

print("dataset_semanal (con NASA POWER):", dataset_semanal.shape)
assert dataset_semanal.shape[0] == 11445, f"ALERTA: cambio el numero de filas: {dataset_semanal.shape[0]}"

print("\narea_ha_total describe (debe ser identico al de antes):")
print(dataset_semanal["area_ha_total"].describe())

print("\nNulos en las nuevas columnas de NASA POWER:")
print(dataset_semanal[cols_nasa].isnull().sum())

dataset_semanal (con NASA POWER): (11445, 21)

area_ha_total describe (debe ser identico al de antes):
count    11445.000000
mean       291.371357
std       1219.044315
min          0.010000
25%         16.140000
50%         45.020000
75%        163.380000
max      55192.060000
Name: area_ha_total, dtype: float64

Nulos en las nuevas columnas de NASA POWER:
T2M     0
RH2M    0
WS2M    0
dtype: int64


In [9]:
print("Muestra de 5 filas con las columnas nuevas de NASA POWER:")
print(dataset_semanal[["municipio_norm", "uf", "semana", "area_ha_total"] + cols_nasa].sample(5, random_state=42))

print("\nEstadisticas de las columnas nuevas:")
print(dataset_semanal[cols_nasa].describe())

Muestra de 5 filas con las columnas nuevas de NASA POWER:
            municipio_norm  uf     semana  area_ha_total        T2M  \
8390                 ACARA  PA 2024-06-03          60.12  26.844286   
4901              TABAPORA  MT 2023-07-10          90.57  26.402857   
9854  PIMENTEIRAS DO OESTE  RO 2024-09-02          92.48  30.540000   
1883  BOM JESUS DAS SELVAS  MA 2022-08-15          17.97  26.678571   
9600                PACAJA  PA 2024-08-19          94.22  30.022857   

           RH2M      WS2M  
8390  93.112857  0.111429  
4901  55.095714  0.077143  
9854  32.590000  0.345714  
1883  67.675714  0.064286  
9600  57.674286  0.031429  

Estadisticas de las columnas nuevas:
                T2M          RH2M          WS2M
count  11445.000000  11445.000000  11445.000000
mean      28.085488     68.733647      0.244149
std        2.687043     18.847557      0.399069
min       18.184286     20.098571      0.000000
25%       25.887143     54.397143      0.055714
50%       27.834286  

In [13]:
dataset_semanal.head()

,semana,municipio_norm,uf,num_alertas,area_ha_total,score_riesgo_promedio,num_incendios,frp_total,frp_promedio,precipitacion_promedio,...,riesgo_fuego_promedio,pm10_reanalise,pm2_5_reanalise,o3_reanalise,no2_reanalise,co_reanalise,so2_reanalise,T2M,RH2M,WS2M
0,2021-12-27,BOCA DO ACRE,AM,1,11.65,4.0,1.0,6.7,6.7,0.4,...,0.0,23.165,18.445,16.250,0.080,0.115,0.02,26.425,90.635,0.020
1,2021-12-27,CANDEIAS DO JAMARI,RO,2,48.69,4.0,0.0,0.0,0.0,NaN,...,NaN,20.620,16.535,16.965,0.195,0.120,0.14,25.365,95.210,0.025
2,2021-12-27,MANOEL URBANO,AC,2,33.45,3.5,0.0,0.0,0.0,NaN,...,NaN,23.300,18.150,11.330,0.065,0.115,0.01,26.165,91.140,0.000
3,2021-12-27,PORTO VELHO,RO,5,131.47,4.6,0.0,0.0,0.0,NaN,...,NaN,20.710,16.615,17.040,0.175,0.120,0.14,25.535,94.720,0.080
4,2021-12-27,SENA MADUREIRA,AC,2,38.57,5.5,0.0,0.0,0.0,NaN,...,NaN,23.650,18.560,12.780,0.065,0.115,0.02,26.300,89.380,0.020


## Paso 4: Resumen final de columnas por dataset (para el diccionario de variables del informe)

In [10]:
print("=== dataset_alertas: columnas finales ===")
for c in dataset_alertas.columns:
    print(" -", c)

print("\n=== dataset_semanal: columnas finales ===")
for c in dataset_semanal.columns:
    print(" -", c)

print(f"\ndataset_alertas: {dataset_alertas.shape[0]} filas x {dataset_alertas.shape[1]} columnas")
print(f"dataset_semanal: {dataset_semanal.shape[0]} filas x {dataset_semanal.shape[1]} columnas")

=== dataset_alertas: columnas finales ===
 - id_alerta
 - gid
 - classname
 - view_date
 - publish_month
 - sensor
 - satellite
 - municipality
 - mun_geocod
 - uf
 - lat
 - lon
 - area_ha
 - areauckm
 - dentro_area_protegida
 - uc
 - score_riesgo
 - Nivel_Riesgo_Amenaza
 - municipio_norm
 - num_incendios_7d_previo
 - frp_total_7d_previo
 - precipitacion_promedio_7d_previo
 - dias_sin_lluvia_promedio_7d_previo
 - riesgo_fuego_promedio_7d_previo
 - pm10_reanalise_7d_previo
 - pm2_5_reanalise_7d_previo
 - o3_reanalise_7d_previo
 - no2_reanalise_7d_previo
 - co_reanalise_7d_previo
 - so2_reanalise_7d_previo
 - T2M_7d_previo
 - RH2M_7d_previo
 - WS2M_7d_previo

=== dataset_semanal: columnas finales ===
 - semana
 - municipio_norm
 - uf
 - num_alertas
 - area_ha_total
 - score_riesgo_promedio
 - num_incendios
 - frp_total
 - frp_promedio
 - precipitacion_promedio
 - dias_sin_lluvia_promedio
 - riesgo_fuego_promedio
 - pm10_reanalise
 - pm2_5_reanalise
 - o3_reanalise
 - no2_reanalise
 - co_

## Paso 5: Guardar los datasets finales completos (DETER + BDQueimadas + SISAM + NASA POWER)

In [11]:
dataset_alertas.to_csv("dataset_alertas_COMPLETO.csv", index=False, encoding="utf-8")
dataset_semanal.to_csv("dataset_semanal_COMPLETO.csv", index=False, encoding="utf-8")

print("Guardado: dataset_alertas_COMPLETO.csv  ->", dataset_alertas.shape)
print("Guardado: dataset_semanal_COMPLETO.csv  ->", dataset_semanal.shape)

Guardado: dataset_alertas_COMPLETO.csv  -> (49971, 33)
Guardado: dataset_semanal_COMPLETO.csv  -> (11445, 21)


## Resumen del pipeline completo (para el informe del Hito 1/2)

| Notebook | Fuente | Que aporta |
|---|---|---|
| 01 | DETER (INPE) | Alertas de deforestacion: fecha, ubicacion, area, tipo de amenaza -> targets de ambos frentes |
| 02 | BDQueimadas (INPE) | Focos de calor, precipitacion, dias sin lluvia, riesgo de fuego (ya calculado por INPE) |
| 03 | Merge DETER + BDQueimadas | Construccion de `dataset_alertas` (Frente 2) y `dataset_semanal` (Frente 1), con ventana movil de 7 dias (sin leakage) y agregado semanal respectivamente |
| 04 | SISAM (INPE/Fiocruz, modelo CAMS) | 6 contaminantes atmosfericos: PM2.5, PM10, O3, NO2, SO2, CO |
| 05 | Merge SISAM | Se agregan los 6 contaminantes a ambos datasets, misma logica de granularidad |
| 06 | NASA POWER | Temperatura, humedad relativa, velocidad de viento (por centroide de municipio) |
| 07 | Merge NASA POWER (este notebook) | Se cierran las variables agrometeorologicas clasicas en ambos datasets |

**Correccion transversal aplicada en todos los merges:** la clave de cruce es siempre `(municipio_norm, uf)`, nunca `municipio_norm` sola, porque existen municipios homonimos en distintos estados de la Amazonia Legal (ej. `PAU D'ARCO` en Para y en Tocantins) que de otro modo mezclarian datos de lugares distintos.



Con `dataset_alertas_COMPLETO.csv` y `dataset_semanal_COMPLETO.csv` el pipeline de datos queda cerrado.

## Diccionario de variables — Datasets finales (DETER + BDQueimadas + SISAM + NASA POWER)

### `dataset_alertas_COMPLETO.csv` (49,971 filas — una fila = una alerta individual)
**Uso:** Frente 2 — Clasificación multiclase del Nivel de Riesgo de Amenaza.

| Variable | Descripción resumida |
|---|---|
| `id_alerta` | Identificador único de la alerta (generado por el equipo, no depende del `gid` de INPE) |
| `gid` | Identificador original de INPE (referencia, no usar como clave única) |
| `classname` | Tipo de amenaza detectada: tala total, quema, degradación, minería, corte selectivo, etc. |
| `view_date` | Fecha exacta en que el satélite detectó la alerta |
| `publish_month` | Mes en que INPE publicó oficialmente la alerta |
| `sensor`, `satellite` | Sensor y satélite que capturaron la imagen |
| `municipality`, `municipio_norm` | Nombre del municipio (original y normalizado sin tildes/mayúsculas) |
| `mun_geocod` | Código oficial IBGE del municipio |
| `uf` | Estado brasileño (abreviatura) — parte de la clave de cruce entre fuentes |
| `lat`, `lon` | Coordenadas del centroide del polígono de la alerta |
| `area_ha` | Hectáreas afectadas por esa alerta puntual |
| `areauckm` | Área (km²) del polígono que cae dentro de una Unidad de Conservación, si aplica |
| `dentro_area_protegida` | 1 si la alerta cae en área protegida, 0 si no |
| `uc` | Nombre de la Unidad de Conservación afectada (vacío si no aplica) |
| `score_riesgo` | Puntaje numérico (1-8) usado para construir el nivel de riesgo |
| **`Nivel_Riesgo_Amenaza`** | **TARGET del Frente 2**: Bajo / Moderado / Alto / Crítico |
| `num_incendios_7d_previo` | Cantidad de focos de calor detectados en los 7 días previos a la alerta, mismo municipio |
| `frp_total_7d_previo` | Intensidad energética total (MW) de esos focos — mide qué tan fuertes fueron los incendios |
| `precipitacion_promedio_7d_previo` | Lluvia promedio (mm) en los 7 días previos |
| `dias_sin_lluvia_promedio_7d_previo` | Días consecutivos sin lluvia, promedio de esa ventana |
| `riesgo_fuego_promedio_7d_previo` | Índice de riesgo de incendio (0-1) ya calculado por INPE |
| `pm10_reanalise_7d_previo`, `pm2_5_reanalise_7d_previo` | Material particulado (contaminación del aire) promedio de los 7 días previos |
| `o3_reanalise_7d_previo`, `no2_reanalise_7d_previo`, `co_reanalise_7d_previo`, `so2_reanalise_7d_previo` | Concentración promedio de ozono, dióxido de nitrógeno, monóxido de carbono y dióxido de azufre en esos 7 días — indicadores indirectos de combustión/quema en la zona |
| `T2M_7d_previo` | Temperatura del aire promedio (°C) de los 7 días previos |
| `RH2M_7d_previo` | Humedad relativa promedio (%) de los 7 días previos |
| `WS2M_7d_previo` | Velocidad del viento promedio (m/s) de los 7 días previos |

---

### `dataset_semanal_COMPLETO.csv` (11,445 filas — una fila = una semana + un municipio)
**Uso:** Frente 1 — Regresión/Series de Tiempo de hectáreas en riesgo.

| Variable | Descripción resumida |
|---|---|
| `semana` | Semana calendario (lunes de esa semana) |
| `municipio_norm`, `uf` | Municipio y estado (clave de cruce entre fuentes) |
| `num_alertas` | Cuántas alertas de deforestación hubo en ese municipio esa semana |
| **`area_ha_total`** | **TARGET del Frente 1**: hectáreas totales afectadas esa semana en ese municipio |
| `score_riesgo_promedio` | Promedio del score de riesgo de todas las alertas de esa semana |
| `num_incendios` | Focos de calor detectados esa semana en ese municipio |
| `frp_total`, `frp_promedio` | Intensidad energética total y promedio de esos focos |
| `precipitacion_promedio` | Lluvia promedio (mm) de la semana |
| `dias_sin_lluvia_promedio` | Días consecutivos sin lluvia, promedio de la semana |
| `riesgo_fuego_promedio` | Índice de riesgo de incendio promedio de la semana (calculado por INPE) |
| `pm10_reanalise`, `pm2_5_reanalise` | Material particulado promedio de la semana |
| `o3_reanalise`, `no2_reanalise`, `co_reanalise`, `so2_reanalise` | Concentración promedio semanal de ozono, NO2, CO y SO2 |
| `T2M` | Temperatura del aire promedio (°C) de la semana |
| `RH2M` | Humedad relativa promedio (%) de la semana |
| `WS2M` | Velocidad del viento promedio (m/s) de la semana |

**Nota metodológica clave:** todas las variables climáticas de `dataset_alertas` usan una ventana de **7 días hacia atrás** desde la fecha exacta de cada alerta (nunca datos futuros, para evitar fuga de información). En `dataset_semanal`, las variables corresponden al promedio de **toda la semana calendario** de esa fila — si se van a usar como predictoras de `area_ha_total` a futuro (horizonte 7/30 días), deben *lagearse* en la fase de Feature Engineering antes de modelar.